# Train Sentiment Analyst in Google Colab

This notebook trains the model, creates an evaluation image, and uploads the outputs to `isravel-eng/SentimentalAnalyst`. Before running, create a GitHub token with repository Contents read/write permission. In Colab's left sidebar, add a secret named `GITHUB_TOKEN`, paste the token, and allow this notebook to use it. Never paste the token in a code cell.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/content/SentimentalAnalyst')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/isravel-eng/SentimentalAnalyst.git', str(REPO_DIR)], check=True)
%cd /content/SentimentalAnalyst
%pip install -q -r requirements.txt

## Train and review

The default trains on a balanced sample of up to 100,000 Sentiment140 tweets. Set `DATASET = 'imdb'` for Keras movie reviews or `DATASET = 'sample'` for the included three-class demo. Sentiment140 and IMDB are binary datasets; neither has neutral training examples. Set `MAX_SAMPLES = 0` to use all Sentiment140 tweets.

In [ ]:
DATASET = 'sentiment140'
MAX_SAMPLES = 100000
if DATASET == 'imdb':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-imdb.txt'], check=True)

subprocess.run([
    sys.executable, 'main.py', 'train', '--dataset', DATASET,
    '--max-samples', str(MAX_SAMPLES),
    '--model', 'models/sentiment_model.pkl',
    '--report', 'reports/evaluation.txt',
    '--image-report', 'reports/evaluation_report.png',
], check=True)

from IPython.display import Image, display
display(Image('reports/evaluation_report.png'))

## Save the model and reports to GitHub

This cell creates or updates the artifact files on the repository's `main` branch. The GitHub token is retrieved from Colab Secrets and is not written to disk or committed.

In [ ]:
import base64
import requests
from google.colab import userdata

OWNER, REPO, BRANCH = 'isravel-eng', 'SentimentalAnalyst', 'main'
TOKEN = userdata.get('GITHUB_TOKEN')
if not TOKEN:
    raise ValueError('Add GITHUB_TOKEN in Colab Secrets and allow this notebook to use it.')
headers = {
    'Authorization': f'Bearer {TOKEN}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28',
}

def upload_to_github(local_path, repo_path):
    local_path = Path(local_path)
    url = f'https://api.github.com/repos/{OWNER}/{REPO}/contents/{repo_path}'
    current = requests.get(url, headers=headers, params={'ref': BRANCH}, timeout=30)
    payload = {
        'message': f'Update training artifact: {repo_path}',
        'content': base64.b64encode(local_path.read_bytes()).decode('ascii'),
        'branch': BRANCH,
    }
    if current.status_code == 200:
        payload['sha'] = current.json()['sha']
    elif current.status_code != 404:
        current.raise_for_status()
    response = requests.put(url, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    print(f'Saved {repo_path} to GitHub')

upload_to_github('models/sentiment_model.pkl', 'models/sentiment_model.pkl')
upload_to_github('reports/evaluation.txt', 'reports/evaluation.txt')
upload_to_github('reports/evaluation_report.png', 'reports/evaluation_report.png')

## Deploy the app

After the upload cell succeeds, create an app on Streamlit Community Cloud using this repository, the `main` branch, and `app.py`. The app will load the model file that Colab just saved.